In [ ]:
import torch
import json
import os
import time
from pathlib import Path
from rfdetr import RFDETRMedium
import traceback

def get_optimal_batch_size():
    """Automatically selects a batch size based on VRAM"""
    if not torch.cuda.is_available():
        return 2
    
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    if vram_gb >= 24:    # RTX 4090, A100
        return 8
    elif vram_gb >= 16:  # RTX 4080, RTX 3080 Ti
        return 6  
    elif vram_gb >= 12:  # RTX 4070 Ti, RTX 3080
        return 4
    elif vram_gb >= 8:   # RTX 4060 Ti, RTX 3070
        return 3
    else:               # RTX 3060, GTX 1660
        return 2

def _validate_coco_split(split, annotation_file):
    with open(annotation_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    for key in ("images", "annotations", "categories"):
        if not isinstance(data.get(key), list):
            raise ValueError(f"{split}: COCO field '{key}' must be a list")

    categories = {category.get("id"): category.get("name") for category in data["categories"]}
    if set(categories.values()) != {"Slipper", "Sneaker"}:
        raise ValueError(f"{split}: expected the English Slipper/Sneaker labels")

    images = {}
    for image in data["images"]:
        image_id = image.get("id")
        width = image.get("width")
        height = image.get("height")
        if image_id in images or not isinstance(width, int) or not isinstance(height, int) or width <= 0 or height <= 0:
            raise ValueError(f"{split}: every image needs a unique positive id, width, and height")
        images[image_id] = (width, height)

    for annotation in data["annotations"]:
        image_id = annotation.get("image_id")
        category_id = annotation.get("category_id")
        bbox = annotation.get("bbox")
        if image_id not in images or category_id not in categories or not isinstance(bbox, list) or len(bbox) != 4:
            raise ValueError(f"{split}: annotation references invalid image/category data")
        x, y, width, height = bbox
        image_width, image_height = images[image_id]
        if (not all(isinstance(value, (int, float)) for value in bbox) or width <= 0 or height <= 0
                or x < 0 or y < 0 or x + width > image_width or y + height > image_height):
            raise ValueError(f"{split}: annotation contains an invalid or out-of-bounds bbox")

    return data

def validate_dataset(dataset_dir):
    """Validate the dataset structure and COCO annotation integrity."""
    dataset_path = Path(dataset_dir)

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset directory does not exist: {dataset_path.resolve()}"
        )

    required_splits = ["train"]
    optional_splits = ["valid", "test"]

    found_splits = []

    for split in required_splits + optional_splits:
        split_dir = dataset_path / split
        ann_file = split_dir / "_annotations.coco.json"

        if split_dir.exists() and ann_file.exists():
            data = _validate_coco_split(split, ann_file)

            n_images = len(data["images"])
            n_annotations = len(data["annotations"])
            n_categories = len(data["categories"])

            found_splits.append(split)

            print(
                f"✅ {split}: {n_images} images, "
                f"{n_annotations} annotations, "
                f"{n_categories} categories"
            )

            print(f"📋 Categories in {split}:")
            for category in data["categories"]:
                count = sum(
                    1
                    for annotation in data["annotations"]
                    if annotation["category_id"] == category["id"]
                )
                print(
                    f"  ID {category['id']}: "
                    f"{category['name']} - {count} objects"
                )

        elif split in required_splits:
            raise FileNotFoundError(f"Missing {split} split: {split_dir}")
        else:
            print(f"ℹ️  {split}: is not available")

    if not found_splits:
        raise ValueError("No valid dataset splits found.")

    return found_splits


In [ ]:
# --- Set training parameters here ---

from pathlib import Path

project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

dataset_path = project_root / "synthetic-shoe-dataset"
output_path = project_root / "synthetic_trained_model_20epochs"

# Training parameters
epochs = 20
batch_size = 1
lr = 1e-5
lr_encoder = 1e-6
weight_decay = 1e-4

# Hardware
device_setting = "cpu"

# Advanced
no_early_stopping = False
patience = 5
warmup_epochs = 2
grad_accum = 1

# Debug
quick_test = False
skip_validation = False

In [ ]:
def train_in_notebook():
    print("🚀 RF-DETR Universal Training Script")
    print("=" * 50)

    # Validate dataset
    if not skip_validation:
        print("\n🔍 Checking dataset...")

        try:
            found_splits = validate_dataset(dataset_path)
        except Exception as error:
            print(f"❌ Dataset error: {error}")
            return

    # Select device
    if device_setting == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = device_setting

    print("\n🖥️  Hardware:")
    print(f"  Device: {device}")

    if device == "cuda":
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU: {gpu_name}")
        print(f"  VRAM: {vram_gb:.1f} GB")

    # Batch size
    final_batch_size = batch_size

    if final_batch_size is None:
        final_batch_size = get_optimal_batch_size()
        print(f"  Auto batch size: {final_batch_size}")
    else:
        print(f"  Manual batch size: {final_batch_size}")

    effective_batch = final_batch_size * grad_accum
    print(f"  Effective batch size: {effective_batch}")

    # Quick test uses one epoch
    final_epochs = 1 if quick_test else epochs

    if quick_test:
        print(f"\n⚡ Quick test mode: {final_epochs} epoch")

    # Initialize model
    print("\n🤖 Initializing RF-DETR Medium...")
    model = RFDETRMedium()

    # Convert Path objects to strings for RF-DETR and JSON
    dataset_dir = str(dataset_path)
    output_dir = str(output_path)

    # Training configuration
    config = {
    "dataset_dir": str(dataset_path),
    "epochs": final_epochs,
    "batch_size": final_batch_size,
    "grad_accum_steps": grad_accum,
    "lr": lr,
    "lr_encoder": lr_encoder,
    "weight_decay": weight_decay,
    "device": device,
    "output_dir": str(output_path),
    "early_stopping": not no_early_stopping,
    "warmup_epochs": warmup_epochs,
    "use_ema": True,
    "checkpoint_interval": 1,
}

    print("\n📋 Training configuration:")

    for key, value in config.items():
        print(f"  {key}: {value}")

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Save JSON-compatible configuration
    config_file = Path(output_dir) / "training_config.json"

    with open(config_file, "w", encoding="utf-8") as file:
        json.dump(config, file, indent=2)

    print(f"\n💾 Configuration saved: {config_file}")

    print("\n🎯 Starting training...")
    print(f"📁 Output: {output_dir}")
    print("-" * 50)

    try:
        model.train(**config)

        print("\n" + "=" * 50)
        print("🎉 Training completed!")
        print(f"📁 Model saved to: {output_dir}")

        checkpoint_best = Path(output_dir) / "checkpoint_best_total.pth"

        if checkpoint_best.exists():
            print(f"🏆 Best model: {checkpoint_best}")
        else:
            print("⚠️ Best checkpoint was not found.")

    except KeyboardInterrupt:
        print("\n⏹️ Training interrupted by user.")

    except Exception as error:
        print(f"\n❌ Training error: {error}")
        traceback.print_exc()


train_in_notebook()